# CellWhisperer Benchmark Summary

Aggregates performance metrics across all CellWhisperer benchmark datasets for a given model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib

In [ ]:
# Parameters
model_name = snakemake.wildcards.model_name
print(f"Creating benchmark summary for model: {model_name}")

In [ ]:
# Load all performance metrics files
performance_data = []

for metrics_file in snakemake.input.performance_files:
    if Path(metrics_file).exists():
        try:
            df = pd.read_csv(metrics_file)
            
            # Extract dataset and metadata_col from file path
            parts = Path(metrics_file).parts
            dataset_idx = parts.index('datasets') + 1
            dataset = parts[dataset_idx]
            metadata_col = parts[dataset_idx + 1]
            
            df['dataset'] = dataset
            df['metadata_col'] = metadata_col
            df['model'] = model_name
            
            performance_data.append(df)
            print(f"Loaded metrics for {dataset}/{metadata_col}")
        except Exception as e:
            print(f"Error loading {metrics_file}: {e}")
    else:
        print(f"File not found: {metrics_file}")

if performance_data:
    combined_df = pd.concat(performance_data, ignore_index=True)
    print(f"Combined performance data shape: {combined_df.shape}")
    print(f"Datasets included: {combined_df['dataset'].unique()}")
else:
    print("No performance data found!")
    combined_df = pd.DataFrame()

In [ ]:
# Save summary CSV
if not combined_df.empty:
    # Ensure output directory exists
    Path(snakemake.output.summary_csv).parent.mkdir(parents=True, exist_ok=True)
    
    combined_df.to_csv(snakemake.output.summary_csv, index=False)
    print(f"Summary CSV saved to: {snakemake.output.summary_csv}")
else:
    # Create empty file to satisfy Snakemake
    Path(snakemake.output.summary_csv).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame().to_csv(snakemake.output.summary_csv, index=False)
    print("Created empty summary CSV (no data available)")

In [ ]:
# Create summary plot
if not combined_df.empty and len(combined_df) > 0:
    # Plot performance metrics
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'CellWhisperer Benchmark Summary - {model_name}', fontsize=16)
    
    # Key metrics to plot
    metrics = ['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted']
    
    for i, metric in enumerate(metrics):
        ax = axes[i // 2, i % 2]
        
        if metric in combined_df.columns:
            # Bar plot of metric by dataset
            data_for_plot = combined_df.groupby('dataset')[metric].mean().sort_values(ascending=False)
            
            bars = ax.bar(range(len(data_for_plot)), data_for_plot.values)
            ax.set_xlabel('Dataset')
            ax.set_ylabel(metric.replace('_', ' ').title())
            ax.set_title(f'{metric.replace("_", " ").title()} by Dataset')
            ax.set_xticks(range(len(data_for_plot)))
            ax.set_xticklabels(data_for_plot.index, rotation=45, ha='right')
            
            # Add value labels on bars
            for bar, value in zip(bars, data_for_plot.values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{value:.3f}', ha='center', va='bottom')
        else:
            ax.text(0.5, 0.5, f'{metric} not available', 
                   transform=ax.transAxes, ha='center', va='center')
    
    plt.tight_layout()
    plt.savefig(snakemake.output.summary_plot, dpi=300, bbox_inches='tight')
    print(f"Summary plot saved to: {snakemake.output.summary_plot}")
else:
    # Create empty plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.text(0.5, 0.5, f'No benchmark data available for {model_name}', 
           transform=ax.transAxes, ha='center', va='center', fontsize=14)
    ax.set_title(f'CellWhisperer Benchmark Summary - {model_name}')
    plt.savefig(snakemake.output.summary_plot, dpi=300, bbox_inches='tight')
    print(f"Empty summary plot saved to: {snakemake.output.summary_plot}")

plt.close('all')

In [ ]:
# Print summary statistics
if not combined_df.empty:
    print("\n=== Benchmark Summary ===")
    print(f"Model: {model_name}")
    print(f"Datasets evaluated: {len(combined_df['dataset'].unique())}")
    
    # Overall performance
    for metric in ['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted']:
        if metric in combined_df.columns:
            mean_val = combined_df[metric].mean()
            std_val = combined_df[metric].std()
            print(f"Mean {metric}: {mean_val:.3f} ± {std_val:.3f}")
    
    # Best performing dataset
    if 'accuracy' in combined_df.columns:
        best_dataset = combined_df.loc[combined_df['accuracy'].idxmax(), 'dataset']
        best_accuracy = combined_df['accuracy'].max()
        print(f"Best performing dataset: {best_dataset} (accuracy: {best_accuracy:.3f})")
    
    print("=" * 25)
else:
    print("No benchmark data to summarize.")